# Data Collection Process (DCP): Multilingual Sentiment Analysis
**CSS 410, Research Tools and Methods**

This notebook runs the whole DCP pipeline stage by stage. The terminal output of every stage is
**shown in the notebook and also saved to `logs/<stage>.log`**, so you can keep it separately.
At the end it builds `results/summary.md` and CSV tables for the report.

**How to use:** put this file in the `dcp/` folder (next to `run_all.py`), open it with the `.venv`
kernel and use *Run All*. Stage 1 downloads about 1 GB, so the first run takes a while.

In [1]:
import os, subprocess, sys, time, datetime
from pathlib import Path
import pandas as pd

DEMO = os.environ.get("DCP_DEMO") == "1"          # DCP_DEMO=1 -> synthetic data, offline test
BASE = Path.cwd() if (Path.cwd() / "run_all.py").exists() else Path.cwd() / "dcp"
assert (BASE / "run_all.py").exists(), "Put the notebook next to run_all.py (the dcp folder)."
os.chdir(BASE)
LOGS, RES = BASE / "logs", BASE / "results"
LOGS.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def run(name, *args):
    """Run one stage, stream its output into the notebook AND into logs/<name>.log"""
    cmd = [sys.executable, "-u", *args]
    header = f"$ {' '.join(map(str, cmd[1:]))}   [{datetime.datetime.now():%Y-%m-%d %H:%M:%S}]"
    print(header)
    t0, lines = time.time(), [header]
    p = subprocess.Popen(cmd, cwd=BASE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, errors="replace")
    for line in p.stdout:
        if "\r" in line:                       # progress bars: keep only the final state
            line = line.split("\r")[-1] if line.split("\r")[-1].strip() else line.split("\r")[-2] + "\n"
        print(line, end=""); lines.append(line.rstrip("\n"))
    rc = p.wait()
    tail = f"[exit code {rc}, {time.time()-t0:.0f}s]"
    print(tail); lines.append(tail)
    (LOGS / f"{name}.log").write_text("\n".join(lines), encoding="utf-8")
    assert rc == 0, f"stage '{name}' failed, see logs/{name}.log"
print("working dir:", BASE, "| demo mode:", DEMO)

working dir: /Users/allikhankoshamet/Downloads/dcp 3 | demo mode: False


## Environment check
The notebook must run on the **same Python that has the dependencies**. This cell installs `requirements.txt`
into the kernel that is running right now (safe to re-run) and prints the versions.

In [2]:
import importlib.util
need = [m for m in ("pandas", "numpy", "datasets", "transformers", "sentencepiece", "datasketch")
        if importlib.util.find_spec(m) is None]
if need:
    print("installing into this kernel:", need)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(BASE / "requirements.txt")])
import pandas, pyarrow, datasketch
print("python:", sys.executable)
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| datasketch", datasketch.__version__)

installing into this kernel: ['sentencepiece', 'datasketch']



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


python: /opt/homebrew/opt/python@3.11/bin/python3.11
pandas 3.0.3 | pyarrow 24.0.0 | datasketch 2.0.0


## 0. Hugging Face access (needed once)
`issai/kazsandra` is a **gated** dataset. Open <https://huggingface.co/datasets/issai/kazsandra>, accept the
conditions, create a token at <https://huggingface.co/settings/tokens> (read access) and run the cell below.
Without it Stage 1 still works for every other source, but there will be **no labelled Kazakh data**.

In [5]:
# NEVER paste the token into the notebook text: the file may be shared. It is asked for here (hidden input)
# and cached by huggingface_hub, so the stage scripts (separate processes) can use it.
from huggingface_hub import get_token, login
from getpass import getpass
if not DEMO and not get_token():
    login(token=getpass("Hugging Face token (input is hidden): ").strip())
print("Hugging Face token available:", bool(get_token()) or DEMO)

Hugging Face token available: True


## 1. Stage 1: multi-source ingestion (open datasets)

In [6]:
run("stage1_collect", "stage1_collect.py", *(["--demo"] if DEMO else []))

$ -u stage1_collect.py   [2026-10-28 02:04:00]
-> sst2  (stanfordnlp/sst2)
  [sst2] columns=['idx', 'sentence', 'label'] -> text=sentence label=label
  label distribution: {'pos': 38013, 'neg': 30208}
  saved 68,221 rows
-> yelp_polarity  (fancyzhx/yelp_polarity)
  [yelp_polarity] columns=['text', 'label'] -> text=text label=label
  label distribution: {'neg': 53197, 'pos': 46803}
  saved 100,000 rows
-> amazon_polarity  (fancyzhx/amazon_polarity)
  [amazon_polarity] columns=['label', 'title', 'content'] -> text=content label=label
  label distribution: {'pos': 51267, 'neg': 48733}
  saved 100,000 rows
-> kazsandra  (issai/kazsandra)
  [kazsandra] columns=['custom_id', 'text', 'label', 'domain'] -> text=text label=label
  [kazsandra] mapped 1-5 ratings -> neg/neu/pos
  label distribution: {'pos': 142637, 'neg': 30164, 'neu': 7262}
  saved 180,063 rows
-> tr_winvoker  (winvoker/turkish-sentiment-analysis-dataset)
  [tr_winvoker] columns=['text', 'label', 'dataset'] -> text=text label=la

## 2. Stage 2: quality filtering and deduplication

In [7]:
run("stage2_clean", "stage2_clean.py")

$ -u stage2_clean.py   [2026-10-28 02:13:02]
            step  rows_after
           input     1118134
       min_words     1072729
bot_like_removed     1071464
       script_ok     1071463
     exact_dedup     1048892
      near_dedup     1042573
lang   source                
en     amazon_polarity            99915
       sst2                       43082
       yelp_polarity              99733
kk     kazsandra                 154292
       kk_multidomain_leipzig    199500
kk-ru  krcs                         605
tr     tr_trsav1                 135082
       tr_winvoker               310364

retention per source (clean / raw):
source
amazon_polarity           0.999
kazsandra                 0.857
kk_multidomain_leipzig    0.998
krcs                      0.977
sst2                      0.632
tr_trsav1                 0.901
tr_winvoker               0.972
yelp_polarity             0.997
[exit code 0, 514s]


## 3. Stage 3: morphology-aware filter (RPSC)
Counts WordPiece sub-tokens per word (`bert-base-multilingual-cased`) and flags words with more than 4.
Rows are only **tagged** (`rpsc_flag`), not deleted.

In [10]:
run("stage3_morph", "stage3_morph.py", *(["--heuristic"] if DEMO else []))

$ -u stage3_morph.py   [2026-10-28 02:29:01]
         rows  mean_fertility  pct_rows_flagged
lang                                           
en     242730           1.274              15.6
kk     353792           2.717              58.5
kk-ru     605           2.472              44.8
tr     445446           2.188              47.3
[exit code 0, 72s]


## 4. Stage 5: splits for zero-shot vs joint training
Set `BINARY = True` if the English source stays binary (SST-2 / Yelp / Amazon have no neutral class).

In [11]:
BINARY = False
run("stage5_split", "stage5_split.py", *(["--binary"] if BINARY else []))

$ -u stage5_split.py   [2026-10-28 02:34:02]
label              (none)     neg    neu     pos
split                                           
cs_probe              605       0      0       0
joint_train             0  136690  16238  235440
source_dev              0   12068      0   12205
source_test             0   12069      0   12204
source_train            0   96546      0   97638
target_dev              0   12458   4975   42540
target_test             0   37376  14925  127621
target_train            0   74751  29851  255241
unlabelled_target  199500       0      0       0
[exit code 0, 20s]


## 5. Stage 4: annotation sheet for the 3 annotators

In [12]:
run("stage4_annotate_make", "stage4_annotate.py", "make", "--n", "60" if DEMO else "600")

$ -u stage4_annotate.py make --n 600   [2026-10-28 02:35:10]
wrote 600 rows -> /Users/allikhankoshamet/Downloads/dcp 3/data/annotation/annotation_sheet.csv
[exit code 0, 6s]


## 6. Sanity checks: read this before trusting the numbers

In [13]:
raw = {f.stem: pd.read_csv(f, keep_default_na=False, dtype=str) for f in sorted(Path("data/raw").glob("*.csv"))}
rows = []
for k, d in raw.items():
    vc = d["label"].replace("", "(none)").value_counts().to_dict()
    rows.append([k, d["lang"].iloc[0], d["tier"].iloc[0], len(d), vc])
src_tab = pd.DataFrame(rows, columns=["source", "lang", "tier", "rows_raw", "label_counts"])
display(src_tab)

problems = []
clean = pd.read_csv("data/clean/clean_morph.csv", keep_default_na=False, dtype=str)
lab = clean[clean["label"] != ""]
if not (lab["lang"] == "kk").any():
    problems.append("No labelled Kazakh data: accept the KazSAnDRA terms and log in to Hugging Face (section 0).")
if "krcs" not in raw or len(raw["krcs"]) == 0:
    problems.append("KRCS code-switching probe set is empty.")
if "neu" not in set(lab.loc[lab["lang"] == "en", "label"]):
    problems.append("English source has no neutral class: use BINARY=True or add a 3-class English source.")
if "tr_winvoker" in raw:
    print("winvoker origins kept:", raw["tr_winvoker"]["domain"].value_counts().head(8).to_dict())
print("\n".join(["PROBLEMS:"] + [" - " + p for p in problems]) if problems else "No structural problems detected.")

,source,lang,tier,rows_raw,label_counts
0,amazon_polarity,en,1,100000,"{'pos': 51267, 'neg': 48733}"
1,kazsandra,kk,2,180063,"{'pos': 142637, 'neg': 30164, 'neu': 7262}"
2,kk_multidomain_leipzig,kk,4,200000,{'(none)': 200000}
3,krcs,kk-ru,3,619,{'(none)': 619}
4,sst2,en,1,68221,"{'pos': 38013, 'neg': 30208}"
5,tr_trsav1,tr,2,150000,"{'pos': 50000, 'neg': 50000, 'neu': 50000}"
6,tr_winvoker,tr,2,319231,"{'pos': 262166, 'neg': 56561, 'neu': 504}"
7,yelp_polarity,en,1,100000,"{'neg': 53197, 'pos': 46803}"


winvoker origins kept: {'urun_yorumlari': 234183, 'HUMIR': 65000, 'tweet-pn': 11055, 'magaza_yorumlari': 8489, 'random': 504}
PROBLEMS:
 - English source has no neutral class: use BINARY=True or add a 3-class English source.


## 7. Results tables (for the report)

In [14]:
funnel = pd.read_csv("data/clean/clean_report.csv")
morph = pd.read_csv("data/clean/morph_stats.csv")
splits = pd.read_csv("data/splits/split_stats.csv")
split_pivot = splits.pivot_table(index="split", columns="label", values="n", aggfunc="sum", fill_value=0)
display(funnel); display(morph); display(split_pivot)

,step,rows_after
0,input,1118134
1,min_words,1072729
2,bot_like_removed,1071464
3,script_ok,1071463
4,exact_dedup,1048892
5,near_dedup,1042573


,lang,rows,mean_fertility,pct_rows_flagged
0,en,242730,1.274,15.6
1,kk,353792,2.717,58.5
2,kk-ru,605,2.472,44.8
3,tr,445446,2.188,47.3


label,(none),neg,neu,pos
split,,,,
cs_probe,605,0,0,0
joint_train,0,136690,16238,235440
source_dev,0,12068,0,12205
source_test,0,12069,0,12204
source_train,0,96546,0,97638
target_dev,0,12458,4975,42540
target_test,0,37376,14925,127621
target_train,0,74751,29851,255241
unlabelled_target,199500,0,0,0


## 8. Export `results/summary.md` and CSV tables

In [ ]:
def to_md(df, index=False):
    df = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(x) for x in r) + " |" for r in df.itertuples(index=False))

src_tab.assign(label_counts=src_tab["label_counts"].astype(str)).to_csv(RES / "table_sources.csv", index=False)
funnel.to_csv(RES / "table_cleaning_funnel.csv", index=False)
morph.to_csv(RES / "table_morphology.csv", index=False)
split_pivot.to_csv(RES / "table_splits.csv")

summary = f"""# Data Collection Results ({datetime.date.today()})

## 1. Sources collected
{to_md(src_tab.assign(label_counts=src_tab['label_counts'].astype(str)))}

## 2. Cleaning funnel (Stage 2)
{to_md(funnel)}

## 3. Morphological fragmentation (Stage 3, tokenizer: bert-base-multilingual-cased)
`mean_fertility` = sub-tokens per word. `pct_rows_flagged` = share of texts with at least one word split into more than 4 sub-tokens.

{to_md(morph)}

## 4. Dataset splits (Stage 5)
{to_md(split_pivot, index=True)}

## 5. Known limitations
""" + "\n".join("- " + p for p in problems or ["none detected by the automatic checks"]) + "\n"
(RES / "summary.md").write_text(summary, encoding="utf-8")
print(summary[:1500], "...")
print("\nsaved:", sorted(p.name for p in RES.iterdir()))

## 9. After annotation: inter-annotator agreement
Fill `a1_polarity`, `a2_polarity`, `a3_polarity` (neg/neu/pos) in `data/annotation/annotation_sheet.csv`, then run the cell below.
Only report a kappa value in the paper after this step.

In [ ]:
SHEET = Path("data/annotation/annotation_sheet.csv")
filled = pd.read_csv(SHEET, keep_default_na=False, dtype=str)
if (filled["a1_polarity"] != "").sum() == 0:
    print("Sheet is not annotated yet, skipping.")
else:
    run("stage4_kappa", "stage4_annotate.py", "kappa", str(SHEET))
    run("stage4_gold", "stage4_annotate.py", "gold", str(SHEET))

## 10. Bundle all terminal logs

In [ ]:
import shutil
shutil.make_archive("dcp_logs_and_results", "zip", root_dir=BASE, base_dir="logs")
print("logs saved in:", LOGS, "->", sorted(p.name for p in LOGS.iterdir()))